In [1]:
# ==================
# IMPORTS
# ==================
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from pathlib import Path
import warnings

In [2]:
warnings.filterwarnings("ignore")

In [3]:
# ==================
# DATA PREPARATION
# ==================
def load_data(path: str) -> pd.DataFrame:
    data_path = Path(path)
    if not data_path.exists():
        raise FileNotFoundError(f"{data_path} does not exist")

    return pd.read_csv(path)

df = load_data("../data/processed_data/processed_merged_data.csv")

In [4]:
# ==================
# SPLIT FEATURES / TARGET
# ==================
TARGET = "economic_stress_score"

x = df.drop(columns=[TARGET])
y = df[TARGET]

In [5]:
# ==================
# CROSS VALIDATION SETUP
# ==================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
# ==================
# EVALUATION FUNCTION
# ==================
def evaluate(model, x, y):
    rmse_scores = -cross_val_score(
        model,
        x, y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    r2_scores = cross_val_score(
        model,
        x, y,
        cv=kf,
        scoring="r2"
    )

    return (
        round(rmse_scores.mean(), 2),
        round(rmse_scores.std(), 2),
        round(r2_scores.mean(), 2),
        round(r2_scores.std(), 2)
    )

In [7]:
# ==================
# MODELS
# ==================
models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "XGBRegressor": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}


# ==================
# MODEL COMPARISON
# ==================
results = []

for name, model in models.items():
    rmse_mean, rmse_std, r2_mean, r2_std = evaluate(model, x, y)
    results.append([
        name,
        rmse_mean,
        rmse_std,
        r2_mean,
        r2_std
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "RMSE_mean", "RMSE_std", "R2_mean", "R2_std"]
)

print("\nMODEL COMPARISON (CROSS VALIDATION)")
print("====================================")
print(results_df)


MODEL COMPARISON (CROSS VALIDATION)
                   Model  RMSE_mean  RMSE_std  R2_mean  R2_std
0       LinearRegression       6.20      0.22     0.83    0.01
1  RandomForestRegressor       4.21      0.11     0.92    0.00
2           XGBRegressor       2.40      0.04     0.97    0.00


In [13]:
final_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
final_model.fit(x, y)


# ==================
# FEATURE IMPORTANCE (LEAKAGE CHECK)
# ==================
fi = pd.Series(final_model.feature_importances_, index=x.columns)
print("\nTOP FEATURES")
print("====================================")
print(fi.sort_values(ascending=False))

0.5941263017725923 0.455461426904507

TOP FEATURES
inflation_unemployment              0.339146
inflation_minus_growth              0.134910
gdp_per_capita                      0.079931
unemployment                        0.072607
food_pressure_score                 0.064253
data_completeness_score             0.063151
income_group_Low_income             0.061191
gdp_growth                          0.040353
latitude                            0.022295
population                          0.020134
income_group_Upper_middle_income    0.015973
longitude                           0.015058
cereal_yield                        0.012579
year                                0.012143
income_group_Lower_middle_income    0.011961
cereal_production_tonnes            0.007600
inflation                           0.006052
dietary_energy_supply_adequacy      0.005666
inflation_squared                   0.005634
food_production_index               0.004537
agricultural_land_pct               0.004205
inco

In [9]:
# ==================
# FINAL MODEL
# ==================

# =========================================================
# MODEL A — WITHOUT ENGINEERED FEATURES
# =========================================================
df_a = df.copy()

drop_cols = [
    "inflation_unemployment",
    "inflation_minus_growth",
    "inflation_squared"
]

df_a = df_a.drop(columns=drop_cols)

x_a = df_a.drop(columns=[TARGET])
y_a = df_a[TARGET]

score_a = evaluate(final_model, x_a, y_a)


# =========================================================
# MODEL B — WITH ENGINEERED FEATURES
# =========================================================
df_b = df.copy()

x_b = df_b.drop(columns=[TARGET])
y_b = df_b[TARGET]

score_b = evaluate(final_model, x_b, y_b)

In [10]:
# ==================
# RESULTS
# ==================
print("\nMODEL A (NO ENGINEERED FEATURES)")
print("=================================")
print(f"RMSE: {score_a[0]:.4f} ± {score_a[1]:.4f}")
print(f"R2  : {score_a[2]:.4f} ± {score_a[3]:.4f}")

print("\nMODEL B (WITH ENGINEERED FEATURES)")
print("===================================")
print(f"RMSE: {score_b[0]:.4f} ± {score_b[1]:.4f}")
print(f"R2  : {score_b[2]:.4f} ± {score_b[3]:.4f}")


MODEL A (NO ENGINEERED FEATURES)
RMSE: 2.4400 ± 0.0300
R2  : 0.9700 ± 0.0000

MODEL B (WITH ENGINEERED FEATURES)
RMSE: 2.4000 ± 0.0400
R2  : 0.9700 ± 0.0000


In [11]:
# ==================
# IMPACT ANALYSIS
# ==================
print("\nIMPACT OF ENGINEERING")
print("=====================")

print(f"Δ RMSE: {score_a[0] - score_b[0]:.4f}")
print(f"Δ R2  : {score_b[2] - score_a[2]:.4f}")


IMPACT OF ENGINEERING
Δ RMSE: 0.0400
Δ R2  : 0.0000
